In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/upi_raw_data.csv")

print(df.shape)      
print(df.dtypes)       
print(df.head(10))
print(df.isnull().sum())   

(40, 4)
Month_Year             str
Banks Live         float64
Txn Volume (Mn)    float64
Txn Value (Cr)     float64
dtype: object
  Month_Year  Banks Live  Txn Volume (Mn)  Txn Value (Cr)
0   Jan-2023       462.0          8131.87      1234098.54
1   Feb 2023       471.0          8333.87      1326153.02
2    03/2023       480.0          8744.60      1434064.80
3     Apr'23       487.0          9003.36             NaN
4   May-2023       493.0          9092.98      1407029.17
5   Jun-2023         NaN          9251.25      1439308.25
6   Jul 2023       510.0          9608.23      1500042.74
7    08/2023       519.0         10209.03      1600538.73
8     Sep'23       529.0         10491.27      1682987.94
9   Oct-2023       538.0         11220.87      1684642.08
Month_Year         0
Banks Live         2
Txn Volume (Mn)    0
Txn Value (Cr)     4
dtype: int64


In [3]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[()]", "", regex=True)
)
print(df.columns.tolist())

['month_year', 'banks_live', 'txn_volume_mn', 'txn_value_cr']


In [4]:
print("Duplicates found:", df.duplicated().sum())
df = df.drop_duplicates()
print("New shape after removing duplicates:", df.shape)

Duplicates found: 1
New shape after removing duplicates: (39, 4)


In [5]:
def parse_date(x):
    x = str(x).strip()
    for fmt in ["%b-%Y", "%b %Y", "%m/%Y", "%b'%y"]:
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT   # if nothing matches, mark as missing

df["date"] = df["month_year"].apply(parse_date)
print(df[df["date"].isna()])   # check nothing failed to parse

Empty DataFrame
Columns: [month_year, banks_live, txn_volume_mn, txn_value_cr, date]
Index: []


In [6]:
df = df.drop(columns=["month_year"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df = df.sort_values("date").reset_index(drop=True)

df.head()

,banks_live,txn_volume_mn,txn_value_cr,date,year,month,month_name
0,462.0,8131.87,1234098.54,2023-01-01,2023,1,Jan
1,471.0,8333.87,1326153.02,2023-02-01,2023,2,Feb
2,480.0,8744.60,1434064.80,2023-03-01,2023,3,Mar
3,487.0,9003.36,NaN,2023-04-01,2023,4,Apr
4,493.0,9092.98,1407029.17,2023-05-01,2023,5,May


In [7]:
for col in ["banks_live", "txn_volume_mn", "txn_value_cr"]:
    df[col] = df[col].astype(str).str.strip()
    df[col] = pd.to_numeric(df[col], errors="coerce")  # turns bad/blank values into NaN

print(df.dtypes)   # confirm all three are now float64/int64, not object

banks_live              float64
txn_volume_mn           float64
txn_value_cr            float64
date             datetime64[us]
year                      int32
month                     int32
month_name                  str
dtype: object


In [8]:
df["txn_value_cr"] = df["txn_value_cr"].interpolate(method="linear")
df["banks_live"] = df["banks_live"].interpolate(method="linear").round()

print(df.isnull().sum())   # should be all zeros now

banks_live       0
txn_volume_mn    0
txn_value_cr     0
date             0
year             0
month            0
month_name       0
dtype: int64


In [9]:
df.to_csv("../data/upi_clean_data.csv", index=False)
df.head()

,banks_live,txn_volume_mn,txn_value_cr,date,year,month,month_name
0,462.0,8131.87,1234098.540,2023-01-01,2023,1,Jan
1,471.0,8333.87,1326153.020,2023-02-01,2023,2,Feb
2,480.0,8744.60,1434064.800,2023-03-01,2023,3,Mar
3,487.0,9003.36,1420546.985,2023-04-01,2023,4,Apr
4,493.0,9092.98,1407029.170,2023-05-01,2023,5,May
